# Cox Regression Analysis for DRKD Risk in Type 2 Diabetes Mellitus

## WORKFLOW:
1. Data Generation (synthetic NHANES-like)
2. Data Integration and Cleaning
3. Feature Engineering
4. Cox Proportional Hazards Model
5. Random Survival Forest (5-fold CV)
6. Model Validation and Assumption Checks
7. **Results Export to Excel**

In [ ]:
# ── Cell 0: Imports ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored, cumulative_dynamic_auc
from sksurv.nonparametric import CensoringDistributionEstimator
from sksurv.util import Surv
from sklearn.model_selection import KFold
from sklearn.utils import resample

print('All imports successful')

In [ ]:
# ── Cell 1: Synthetic data generation ─────────────────────────────────────
# NOTE: Replace this cell with real SAIL / Birmingham data when available.

np.random.seed(42)
n_patients = 5000
seqn = np.arange(100000, 100000 + n_patients)

demo = pd.DataFrame({
    'SEQN':     seqn,
    'RIDAGEYR': np.random.normal(55, 15, n_patients).clip(18, 85).astype(int),
    'RIAGENDR': np.random.choice([1, 2], n_patients),
    'RIDRETH3': np.random.choice([1, 2, 3, 4, 6, 7], n_patients),
})
biol = pd.DataFrame({'SEQN': seqn, 'LBXGH':    np.random.normal(6.5, 2.0, n_patients).clip(4.0, 15.0)})
bp   = pd.DataFrame({'SEQN': seqn, 'BPXSY1':   np.random.normal(130, 20, n_patients).clip(90, 200)})
bmx  = pd.DataFrame({'SEQN': seqn, 'BMXBMI':   np.random.normal(28, 6, n_patients).clip(15, 60)})
alb  = pd.DataFrame({'SEQN': seqn, 'URDACT':   np.random.lognormal(2, 1.5, n_patients).clip(0.1, 1000)})
mort = pd.DataFrame({'SEQN': seqn,
                     'MORTSTAT':   np.random.choice([0, 1], n_patients, p=[0.85, 0.15]),
                     'PERMTH_EXM': np.random.exponential(120, n_patients).clip(1, 240).astype(int)})
diab = pd.DataFrame({'SEQN': seqn, 'DIQ010': np.random.choice([1, 2, 3, 9], n_patients, p=[0.15, 0.75, 0.05, 0.05])})

print(f'Synthetic data generated: {n_patients} patients')

In [ ]:
# ── Cell 2: Merge, rename, filter ─────────────────────────────────────────
df = (demo
      .merge(biol, on='SEQN', how='left')
      .merge(bp,   on='SEQN', how='left')
      .merge(bmx,  on='SEQN', how='left')
      .merge(alb,  on='SEQN', how='left')
      .merge(mort, on='SEQN', how='left')
      .merge(diab, on='SEQN', how='left')
)

df = df.rename(columns={
    'RIDAGEYR': 'AGE', 'RIAGENDR': 'SEX', 'RIDRETH3': 'ETHNICITY',
    'LBXGH': 'HBA1C', 'BPXSY1': 'SBP', 'BMXBMI': 'BMI',
    'URDACT': 'UACR', 'MORTSTAT': 'DIED', 'PERMTH_EXM': 'FOLLOW_UP_MONTHS'
})

# T2D patients only
df = df[df['DIQ010'] == 1].copy()

# Feature engineering
df['HBA1C_MMOL']     = (df['HBA1C'] - 2.15) * 10.929
df['LOG_UACR']       = np.log10(df['UACR'] + 1)
df['FOLLOW_UP_YEARS']= df['FOLLOW_UP_MONTHS'] / 12
df['EVENT']          = df['DIED'].fillna(0).astype(int)

# Clinical range filters
df = df[
    (df['HBA1C_MMOL'].between(20, 195)) &
    (df['BMI'].between(14, 70)) &
    (df['SBP'].between(60, 280)) &
    (df['AGE'] >= 18)
]

df = df.dropna(subset=['FOLLOW_UP_YEARS', 'EVENT', 'AGE', 'SEX',
                        'BMI', 'SBP', 'HBA1C_MMOL', 'LOG_UACR', 'ETHNICITY'])

print(f'Final analytic sample: {len(df)} patients')
print(f'Mortality rate: {df["EVENT"].mean()*100:.1f}%')

In [ ]:
# ── Cell 3: EDA ────────────────────────────────────────────────────────────
print('--- Descriptive statistics ---')
print(df[['AGE','SEX','BMI','SBP','HBA1C_MMOL','LOG_UACR','FOLLOW_UP_YEARS','EVENT']].describe().round(2))

vars_to_plot = ['AGE', 'BMI', 'SBP', 'HBA1C_MMOL', 'LOG_UACR', 'FOLLOW_UP_YEARS']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, var in zip(axes.flatten(), vars_to_plot):
    sns.histplot(df[var], ax=ax, kde=True, bins=30)
    ax.set_title(var)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, var in zip(axes.flatten(), vars_to_plot):
    sns.boxplot(x='EVENT', y=var, data=df, ax=ax)
    ax.set_title(f'{var} by event status')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4: Kaplan-Meier curves ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Kaplan-Meier Survival Curves', fontsize=15)

# By HbA1c tertile
df['HBA1C_TERTILE'] = pd.qcut(df['HBA1C_MMOL'], q=3, labels=['Low', 'Medium', 'High'])
ax = axes[0]
for label, color in zip(['Low', 'Medium', 'High'], ['green', 'orange', 'red']):
    mask = df['HBA1C_TERTILE'] == label
    kmf = KaplanMeierFitter()
    kmf.fit(df.loc[mask, 'FOLLOW_UP_YEARS'], event_observed=df.loc[mask, 'EVENT'], label=label)
    kmf.plot_survival_function(ax=ax, ci_show=True)
ml = multivariate_logrank_test(df['FOLLOW_UP_YEARS'], df['HBA1C_TERTILE'], df['EVENT'])
ax.set_title(f'By HbA1c Tertile\nLog-rank p={ml.p_value:.4f}')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Survival probability')

# By Sex
ax = axes[1]
for sex_val, label in [(1, 'Male'), (2, 'Female')]:
    mask = df['SEX'] == sex_val
    kmf = KaplanMeierFitter()
    kmf.fit(df.loc[mask, 'FOLLOW_UP_YEARS'], event_observed=df.loc[mask, 'EVENT'], label=label)
    kmf.plot_survival_function(ax=ax, ci_show=True)
lr = logrank_test(
    df.loc[df['SEX']==1,'FOLLOW_UP_YEARS'], df.loc[df['SEX']==2,'FOLLOW_UP_YEARS'],
    event_observed_A=df.loc[df['SEX']==1,'EVENT'], event_observed_B=df.loc[df['SEX']==2,'EVENT']
)
ax.set_title(f'By Sex\nLog-rank p={lr.p_value:.4f}')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Survival probability')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 5: Cox PH model ───────────────────────────────────────────────────
# Interaction terms (clinically motivated)
df['HBA1C_x_AGE']  = df['HBA1C_MMOL'] * df['AGE']
df['HBA1C_x_UACR'] = df['HBA1C_MMOL'] * df['LOG_UACR']
df['SBP_x_UACR']   = df['SBP']        * df['LOG_UACR']

# Ethnicity dummies
eth_dummies = pd.get_dummies(df['ETHNICITY'].astype(int), prefix='ETH', drop_first=True)
df_cox = pd.concat([df, eth_dummies], axis=1)

main_effects = ['AGE', 'SEX', 'BMI', 'SBP', 'HBA1C_MMOL', 'LOG_UACR']
interactions = ['HBA1C_x_AGE', 'HBA1C_x_UACR', 'SBP_x_UACR']
eth_cols     = list(eth_dummies.columns)
cox_vars     = main_effects + interactions + eth_cols + ['FOLLOW_UP_YEARS', 'EVENT']

cph = CoxPHFitter()
cph.fit(df_cox[cox_vars], duration_col='FOLLOW_UP_YEARS', event_col='EVENT')

print('=== COX MODEL RESULTS ===')
print(cph.summary.to_string())
print(f'\nC-statistic: {cph.concordance_index_:.3f}')

# PH assumption check
cph.check_assumptions(df_cox[cox_vars], show_plots=True)

# Bootstrap 95% CI for C-statistic
np.random.seed(42)
c_boot = []
for _ in range(200):
    idx = resample(range(len(df_cox)), replace=True)
    boot_df = df_cox.iloc[idx]
    try:
        cph_b = CoxPHFitter()
        cph_b.fit(boot_df[cox_vars], duration_col='FOLLOW_UP_YEARS', event_col='EVENT')
        c_boot.append(cph_b.concordance_index_)
    except Exception:
        pass
ci_lower, ci_upper = np.percentile(c_boot, [2.5, 97.5])
print(f'\nCox C-statistic: {cph.concordance_index_:.3f} (95% Bootstrap CI: {ci_lower:.3f}–{ci_upper:.3f}, n={len(c_boot)} resamples)')

In [ ]:
# ── Cell 6: Time-dependent AUC (Cox) ──────────────────────────────────────
y_all        = Surv.from_arrays(df_cox['EVENT'].astype(bool), df_cox['FOLLOW_UP_YEARS'])
risk_scores  = cph.predict_partial_hazard(df_cox[cox_vars]).values

cens_est   = CensoringDistributionEstimator().fit(y_all)
cens_probs = cens_est.predict_proba(df_cox['FOLLOW_UP_YEARS'].values)
safe_mask  = cens_probs > 0

y_safe    = y_all[safe_mask]
risk_safe = risk_scores[safe_mask]

safe_event_times = df_cox.loc[safe_mask & (df_cox['EVENT']==1), 'FOLLOW_UP_YEARS'].values
times = np.percentile(safe_event_times, [25, 50, 75])

auc_cox, mean_auc_cox = cumulative_dynamic_auc(y_all, y_safe, risk_safe, times)

print('=== TIME-DEPENDENT AUC (Cox) ===')
for t, a in zip(times, auc_cox):
    print(f'  t={t:.2f} yr  AUC: {a:.3f}')
print(f'  Mean AUC: {mean_auc_cox:.3f}')

plt.figure(figsize=(7, 5))
plt.plot(times, auc_cox, marker='o', color='steelblue', label='Cox model')
plt.axhline(0.5, linestyle='--', color='grey', label='Random (0.5)')
plt.ylim(0.4, 1.0)
plt.xlabel('Time (years)')
plt.ylabel('AUC')
plt.title('Time-dependent AUC — Cox Model')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: Random Survival Forest with 5-fold CV ─────────────────────────
rf_features = ['AGE', 'SEX', 'BMI', 'SBP', 'HBA1C_MMOL', 'LOG_UACR']
X_rf = df[rf_features].values
y_rf = Surv.from_arrays(df['EVENT'].astype(bool), df['FOLLOW_UP_YEARS'])

print('=== RSF 5-FOLD CROSS-VALIDATED C-INDEX ===')
kf   = KFold(n_splits=5, shuffle=True, random_state=42)
cv_c = []
for fold, (tr, te) in enumerate(kf.split(X_rf), 1):
    rsf_fold = RandomSurvivalForest(n_estimators=100, min_samples_split=10,
                                    min_samples_leaf=5, random_state=42, n_jobs=-1)
    rsf_fold.fit(X_rf[tr], y_rf[tr])
    scores = rsf_fold.predict(X_rf[te])
    c = concordance_index_censored(y_rf[te]['event'], y_rf[te]['time'], scores)[0]
    cv_c.append(c)
    print(f'  Fold {fold}: C-index = {c:.3f}')

cv_mean, cv_std = np.mean(cv_c), np.std(cv_c)
print(f'\nRSF CV C-index: {cv_mean:.3f} ± {cv_std:.3f}')

# Refit on full data for AUC comparison
rsf = RandomSurvivalForest(n_estimators=100, min_samples_split=10,
                           min_samples_leaf=5, random_state=42, n_jobs=-1)
rsf.fit(X_rf, y_rf)
in_sample_c = concordance_index_censored(
    df['EVENT'].astype(bool), df['FOLLOW_UP_YEARS'], rsf.predict(X_rf)
)[0]

# CV time-dependent AUC for RSF
times_rsf = np.array([1, 3, 5, 7])
times_rsf = times_rsf[times_rsf < df['FOLLOW_UP_YEARS'].max()]
cv_aucs = []
for tr, te in KFold(n_splits=5, shuffle=True, random_state=42).split(X_rf):
    rsf_fold = RandomSurvivalForest(n_estimators=100, min_samples_split=10,
                                    min_samples_leaf=5, random_state=42, n_jobs=-1)
    rsf_fold.fit(X_rf[tr], y_rf[tr])
    try:
        auc, _ = cumulative_dynamic_auc(y_rf[tr], y_rf[te], rsf_fold.predict(X_rf[te]), times_rsf)
        cv_aucs.append(auc)
    except Exception:
        pass

mean_auc_rsf = np.mean(cv_aucs, axis=0) if cv_aucs else None

print('\n=== MODEL COMPARISON ===')
print(f'Cox C-statistic (full data):           {cph.concordance_index_:.3f} (95% CI: {ci_lower:.3f}–{ci_upper:.3f})')
print(f'RSF in-sample C-statistic (inflated):  {in_sample_c:.3f}  ← overfitting')
print(f'RSF cross-validated C-statistic:       {cv_mean:.3f} ± {cv_std:.3f}  ← fair comparison')

if mean_auc_rsf is not None:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(times, auc_cox, marker='o', label='Cox model')
    ax.plot(times_rsf, mean_auc_rsf, marker='s', label='RSF (cross-validated)')
    ax.axhline(0.5, linestyle='--', color='grey', label='Random (0.5)')
    ax.set_ylim(0.4, 1.0)
    ax.set_xlabel('Time (years)')
    ax.set_ylabel('AUC')
    ax.set_title('Time-dependent AUC: Cox vs RSF')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 8: RESULTS EXPORT TO EXCEL ───────────────────────────────────────
# Exports all key results into a single downloadable Excel workbook
# with separate sheets for each result table.

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
import warnings
warnings.filterwarnings('ignore')

output_path = 'cox_regression_results.xlsx'

# ── Helper to style a sheet header row ────────────────────────────────────
def style_header(ws, row=1):
    for cell in ws[row]:
        cell.font = Font(bold=True, color='FFFFFF', name='Arial', size=10)
        cell.fill = PatternFill('solid', start_color='1F4E79')
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def style_rows(ws, start=2):
    side = Side(style='thin', color='CCCCCC')
    brd  = Border(left=side, right=side, top=side, bottom=side)
    for i, row in enumerate(ws.iter_rows(min_row=start), start=0):
        for cell in row:
            cell.font = Font(name='Arial', size=10)
            cell.border = brd
            if i % 2 == 0:
                cell.fill = PatternFill('solid', start_color='EBF3FB')

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # ── Sheet 1: Sample Descriptive Statistics ─────────────────────────────
    desc = df[['AGE','SEX','BMI','SBP','HBA1C_MMOL','LOG_UACR','FOLLOW_UP_YEARS','EVENT']].describe().round(3).T
    desc.columns = ['Count','Mean','SD','Min','25th pct','Median','75th pct','Max']
    desc.index.name = 'Variable'
    desc.reset_index().to_excel(writer, sheet_name='1_Descriptive_Stats', index=False)

    # ── Sheet 2: Cox Model Summary ─────────────────────────────────────────
    cox_summary = cph.summary.copy().reset_index()
    cox_summary.columns = [str(c) for c in cox_summary.columns]
    cox_summary = cox_summary.round(4)
    cox_summary.to_excel(writer, sheet_name='2_Cox_Model_Results', index=False)

    # ── Sheet 3: Cox Model Performance ────────────────────────────────────
    perf_rows = [
        ('Cox C-statistic',                   round(cph.concordance_index_, 3)),
        ('Cox 95% CI Lower (Bootstrap)',       round(ci_lower, 3)),
        ('Cox 95% CI Upper (Bootstrap)',       round(ci_upper, 3)),
        ('Bootstrap resamples',                len(c_boot)),
        ('RSF CV C-statistic (mean)',          round(cv_mean, 3)),
        ('RSF CV C-statistic (SD)',            round(cv_std, 3)),
        ('RSF in-sample C-statistic (inflated)', round(in_sample_c, 3)),
    ]
    perf_df = pd.DataFrame(perf_rows, columns=['Metric', 'Value'])
    perf_df.to_excel(writer, sheet_name='3_Model_Performance', index=False)

    # ── Sheet 4: Time-dependent AUC ───────────────────────────────────────
    auc_rows = []
    for t, a in zip(times, auc_cox):
        auc_rows.append({'Time (years)': round(t, 2), 'Model': 'Cox', 'AUC': round(a, 3)})
    if mean_auc_rsf is not None:
        for t, a in zip(times_rsf, mean_auc_rsf):
            auc_rows.append({'Time (years)': round(t, 2), 'Model': 'RSF (CV)', 'AUC': round(a, 3)})
    auc_df = pd.DataFrame(auc_rows)
    auc_df.to_excel(writer, sheet_name='4_Time_Dependent_AUC', index=False)

    # ── Sheet 5: RSF Cross-Validated C-indices ────────────────────────────
    rsf_cv_df = pd.DataFrame({
        'Fold':    [f'Fold {i+1}' for i in range(len(cv_c))],
        'C-index': [round(c, 3) for c in cv_c]
    })
    rsf_cv_df.loc[len(rsf_cv_df)] = ['Mean ± SD', f'{cv_mean:.3f} ± {cv_std:.3f}']
    rsf_cv_df.to_excel(writer, sheet_name='5_RSF_CV_Results', index=False)

    # ── Sheet 6: Descriptive stats by event status ─────────────────────────
    grp = df.groupby('EVENT')[['AGE','BMI','SBP','HBA1C_MMOL','LOG_UACR','FOLLOW_UP_YEARS']].agg(['mean','std']).round(3)
    grp.index = ['No event (0)', 'Event (1)']
    grp.index.name = 'Event status'
    grp.reset_index().to_excel(writer, sheet_name='6_Stats_by_Event', index=False)

# ── Apply styling ──────────────────────────────────────────────────────────
wb = load_workbook(output_path)
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    style_header(ws)
    style_rows(ws)
    for col in ws.columns:
        max_len = max((len(str(cell.value)) if cell.value else 0) for cell in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 40)
wb.save(output_path)

print(f'Results exported to: {output_path}')
print('Sheets included:')
print('  1. Descriptive statistics')
print('  2. Cox model results (HR, CI, p-values)')
print('  3. Model performance (C-statistic, 95% CI)')
print('  4. Time-dependent AUC (Cox and RSF)')
print('  5. RSF cross-validated C-indices by fold')
print('  6. Descriptive statistics by event status')